# φ-Register: Zeckendorf Pruning on GPU

**Runtime → Change runtime type → T4 GPU** (if not already set)

This notebook runs the `zeckendorf-prune` quickstart pipeline with GPU acceleration: dense fine-tune → prune → verify → mask-aware fine-tune → Fibonacci encoding → export.
CIFAR-10 stays on the GPU and is resized there, and training and evaluation run under fp16 autocast in channels-last layout: the T4's tensor cores (65 TFLOPS) take fp16, while fp32 runs on its CUDA cores (8 TFLOPS).

## 0. Install

In [ ]:
# Option A: pip install straight from GitHub
# !pip install -q git+https://github.com/ezexe/zeckendorf-prune.git

# Option B: Upload the package zip, then:
# from google.colab import files
# uploaded = files.upload()  # upload zeckendorf-prune.zip
# !unzip -q zeckendorf-prune.zip -d zeckendorf-prune
# !pip install -q zeckendorf-prune/

# Option C: Clone from GitHub; a re-run pulls new commits into the existing clone.
# A regular install is importable in this running kernel. An editable (-e) install registers the package
# through a .pth file, which Python reads only at startup, so the import in Setup would fail until the
# runtime restarts.
!test -d zeckendorf-prune || git clone -q --depth 1 https://github.com/ezexe/zeckendorf-prune.git
!git -C zeckendorf-prune pull -q --ff-only
!pip install -q ./zeckendorf-prune

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("No GPU: Runtime → Change runtime type → T4 GPU, then run all cells again")

## 1. Setup

Imports and a check that the installed package has `finetune(amp=...)`, config knobs, then the GPU `evaluate` helper.

In [ ]:
import inspect

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from zeckendorf_prune import prune, finetune, check
from zeckendorf_prune.encoding import FibonacciEncoder
from zeckendorf_prune.export import save_checkpoint

if "amp" not in inspect.signature(finetune).parameters:
    raise RuntimeError("installed zeckendorf-prune predates finetune(amp=...): install a newer one "
                       "(GitHub main once that change is pushed) and restart the runtime")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"        # fp16 autocast + loss scaling; T4 tensor cores have no bf16/TF32
torch.backends.cudnn.benchmark = True  # fixed input shapes, so cuDNN's per-shape autotuning pays off
torch.manual_seed(0)

IMG_SIZE = 224                    # ImageNet weights expect ~224 px; 160/128 is faster, less accurate
TRAIN_BATCH = 128                 # as in the original notebook; rescale the learning rates if you change it
EVAL_BATCH = 512                  # eval keeps no activations for backward, so its batches can be larger
DENSE_EPOCHS, DENSE_LR = 2, 0.01  # trains the new 10-class head (and the backbone) before pruning
FT_EPOCHS, FT_LR = 5, 0.001       # mask-aware recovery after pruning, as in the quickstart
print(f"Device: {device}, AMP: {USE_AMP}")

In [ ]:
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=USE_AMP)


def evaluate(model, loader):
    model.eval()
    with torch.inference_mode(), torch.autocast(**AMP):
        correct, total = torch.zeros((), dtype=torch.long, device=device), 0
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum()
            total += len(y)
    return 100.0 * correct.item() / total

## 2. Load model & data

In [ ]:
# Pretrained ResNet-18, swap head for CIFAR-10
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(512, 10)
model = model.to(device, memory_format=torch.channels_last)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# CIFAR-10 stays on the GPU as uint8 (~184 MB) and each batch is normalized, flipped and resized there.
# This replaces the per-image PIL Resize/ToTensor/Normalize that ran in 2 DataLoader workers, and the
# host-to-device copy of every batch at 224 px float32 (77 MB per 128 images).
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


class GPULoader:
    def __init__(self, dataset, batch_size, train):
        self.x = torch.from_numpy(dataset.data).to(device).permute(0, 3, 1, 2).contiguous()  # N,3,32,32
        self.y = torch.as_tensor(dataset.targets, device=device)
        self.batch_size, self.train = batch_size, train

    def __len__(self):
        return -(-len(self.y) // self.batch_size)

    def __iter__(self):
        n = len(self.y)
        order = torch.randperm(n, device=device) if self.train else torch.arange(n, device=device)
        for i in range(0, n, self.batch_size):
            idx = order[i:i + self.batch_size]
            x = self.x[idx].float().div_(255).sub_(MEAN).div_(STD)
            if self.train:  # random horizontal flip
                flip = torch.rand(len(idx), device=device) < 0.5
                x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
            x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            yield x.contiguous(memory_format=torch.channels_last), self.y[idx]


trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)
train_loader = GPULoader(trainset, TRAIN_BATCH, train=True)
test_loader = GPULoader(testset, EVAL_BATCH, train=False)
print(f"Train: {len(trainset)}, Test: {len(testset)}")

## 3. Dense baseline

The pretrained backbone gets a new, randomly initialized 10-class `fc`, so the dense model scores ~10% (chance) until that head is trained.
Pruning is measured against the dense model after `DENSE_EPOCHS` of fine-tuning.

In [ ]:
%%time
dense_results = finetune(model, train_loader, epochs=DENSE_EPOCHS, lr=DENSE_LR, device=device,
                         val_loader=test_loader, amp=USE_AMP)
dense_acc = dense_results["best_val_acc"]
print(f"Dense: {dense_acc:.2f}%")

## 4. Prune & verify

Only conv layers are pruned, so the 10-class `fc` head stays dense.
Pruning `fc.weight` along its 10 output rows would keep at most 5 of them (no two adjacent), leaving the other classes with a bias-only, input-independent logit, which caps test accuracy at 60%.
`prune()` leaves the head dense by default once the package has the `prune_head` parameter; the explicit `layer_types` also covers versions without it.
The ResNet-20 experiment in `.docs/experiment/Zeckendorf.py` prunes conv layers only, too.

In [ ]:
pruned_model, masks = prune(model, inplace=False, layer_types=(nn.Conv2d,))
stats = pruned_model._zeck_prune_stats
print(f"Density (conv weights): {stats['density']:.1%}")
print(f"Pruned layers: {stats['pruned_layers']}")

report = check(pruned_model, masks)
print(f"All masks valid: {report['_summary']['all_masks_valid']}")
print(f"All zeros enforced: {report['_summary']['all_zeros_enforced']}")

In [ ]:
# Accuracy after pruning, before fine-tune
pruned_acc = evaluate(pruned_model, test_loader)
print(f"Pruned (no fine-tune): {pruned_acc:.2f}%")

## 5. Fine-tune

`finetune(..., amp=USE_AMP)` runs the package's mask-aware fine-tune under fp16 autocast with loss scaling on the GPU.
Raise `FT_EPOCHS` for better recovery.

In [ ]:
%%time
ft_results = finetune(pruned_model, train_loader, epochs=FT_EPOCHS, masks=masks, lr=FT_LR, device=device,
                      val_loader=test_loader, amp=USE_AMP)
print(f"Best val accuracy: {ft_results['best_val_acc']:.2f}%")

## 6. Fibonacci encoding

In [ ]:
encoder = FibonacciEncoder(n_digits=10)  # 144 levels
print(f"Grid: {encoder.n_levels} levels, max value: {encoder.max_value}")

sample_param = next(n for n, p in pruned_model.named_parameters() if n in masks)
param = dict(pruned_model.named_parameters())[sample_param]
encoded, scale, rmse = encoder.encode_tensor(param.data, mask=masks[sample_param])
print(f"Sample layer ({sample_param}): RMSE = {rmse:.6f}")

## 7. Summary & export

In [ ]:
report = check(pruned_model, masks)  # re-verify after fine-tuning, not only right after pruning
rows = [
    (f"Dense ({DENSE_EPOCHS} ep ft)", f"{dense_acc:.2f}%"),
    ("Pruned (no ft)", f"{pruned_acc:.2f}%"),
    (f"Pruned ({FT_EPOCHS} ep ft)", f"{ft_results['best_val_acc']:.2f}%"),
    ("Drop vs dense", f"{dense_acc - ft_results['best_val_acc']:.2f} pts"),
    ("Density (conv weights)", f"{stats['density']:.1%}"),
    ("Masks valid", report["_summary"]["all_masks_valid"]),
    ("Zeros enforced", report["_summary"]["all_zeros_enforced"]),
    ("Encoding levels", encoder.n_levels),
]
print("=" * 50)
for label, value in rows:
    print(f"  {label + ':':<24}{value}")
print("=" * 50)

save_checkpoint(pruned_model, masks, "model_pruned.pt", encoder=encoder, metadata={
    "dense_acc": dense_acc, "pruned_acc": pruned_acc,
    "finetuned_acc": ft_results["best_val_acc"], "img_size": IMG_SIZE,
})
print("Saved model_pruned.pt")

In [ ]:
# Download checkpoint locally
from google.colab import files
files.download('model_pruned.pt')